## 0. Install & Imports

In [1]:
# Packages install
%pip install transformers timm underthesea sacrebleu rouge-score bert-score -q
%pip install sentencepiece protobuf -q
%pip install datasets pillow -q
%pip install nltk anthropic -q

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Library import

# Utility
import re, json, time, math, copy, random, warnings, gc, os
from pathlib import Path
from collections import Counter
from typing import List, Dict, Tuple, Optional
from tqdm.auto import tqdm
import timm

# Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Torch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.cuda.amp import autocast, GradScaler
from torchvision import transforms as T

# Transformers
from transformers import (
    AutoTokenizer, AutoModel,
    get_cosine_schedule_with_warmup
)

warnings.filterwarnings('ignore')

In [3]:
# Metrics
from sacrebleu.metrics import BLEU
from rouge_score import rouge_scorer
import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
from nltk.translate.meteor_score import meteor_score as nltk_meteor
import anthropic

try:
    import bert_score
    HAS_BERTSCORE = True
except ImportError:
    HAS_BERTSCORE = False
    print("bert_score not available — BERTScore will be skipped.", flush=True)

## 1. Configuration

### Pipeline Control — Change These Per Notebook Run

| Flag | Description |
|------|-------------|
| `MODEL_VARIANT` | `"A1"` = LSTM decoder; `"A2"` = Transformer decoder |
| `SKIP_TRAIN` | `True` = load checkpoint & go to eval; `False` = train from scratch |

**Variant map:**
```
A1 = ViT + PhoBERT + Co-Attention + LSTM Decoder
A2 = ViT + PhoBERT + Co-Attention + Transformer Decoder
```

In [4]:
# Pipeline control
MODEL_VARIANT = "A1"   # "A1" = LSTM decoder | "A2" = Transformer decoder

# Resolve & print
VARIANT_DESCRIPTIONS = {
    "A1": "ViT + PhoBERT + Co-Attention + LSTM Decoder",
    "A2": "ViT + PhoBERT + Co-Attention + Transformer Decoder",
}

In [5]:
SEED = 42

SKIP_TRAIN    = False  # True = load checkpoint & eval only

NUM_WORKERS = min(4, os.cpu_count() or 1)

LLM_JUDGE_EVAL = False  # Set True to run LLM-as-a-judge (costs API credits)

In [6]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}", flush=True)

if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    vram  = props.total_memory / (1024**3)
    print(f"GPU    : {props.name}  ({vram:.1f} GB VRAM)", flush=True)
    if vram < 6:
        print("  [!] <6 GB VRAM detected — BATCH_SIZE=8 + GradAccum=4 is recommended.", flush=True)

Device : cuda
GPU    : NVIDIA GeForce RTX 3050 Ti Laptop GPU  (4.0 GB VRAM)
  [!] <6 GB VRAM detected — BATCH_SIZE=8 + GradAccum=4 is recommended.


In [7]:
# Data
# Category Filter
TARGET_CATEGORY = "kien_truc"   # None = toàn dataset; "kien_truc" = chỉ kiến trúc

# Limit
MAX_QUESTION_LEN  = 64      # max PhoBERT token length for questions
MAX_ANSWER_LEN    = 15      # max answer tokens (decoder output)
MAX_ANSWER_VOCAB  = 3000    # top-K answers kept as classification labels
MAX_ANSWER_WORDS  = 10       # filter out QA pairs where answer has more than N words

In [8]:
# Image
IMAGE_SIZE = 224     # ViT expects 224x224

# ImageNet normalisation
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Image augmentation
TRAIN_TRANSFORM = T.Compose([
    T.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
    T.RandomCrop(IMAGE_SIZE),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

EVAL_TRANSFORM = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [9]:
# Model dimensions
VIT_MODEL         = "vit_base_patch16_224"   # timm model name
PHOBERT_MODEL     = "vinai/phobert-base-v2"  # HuggingFace model name
VIT_FEAT_DIM      = 768    # ViT-Base hidden dim
PHOBERT_FEAT_DIM  = 768    # PhoBERT-Base hidden dim
CO_ATTN_DIM       = 512    # co-attention hidden dim
FUSION_DIM        = 512    # post-fusion projected dim

# LSTM Decoder (A1 only)
LSTM_HIDDEN_DIM   = 512
LSTM_NUM_LAYERS   = 2
LSTM_DROPOUT      = 0.3

# Transformer Decoder (A2 only)
TRANSF_NHEAD      = 8
TRANSF_NUM_LAYERS = 4
TRANSF_FF_DIM     = 2048
TRANSF_DROPOUT    = 0.1

In [10]:
# Training
BATCH_SIZE              = 32      # safe for 4 GB VRAM
GRADIENT_ACCUMULATION   = 4      # effective batch = BATCH_SIZE * GRADIENT_ACCUMULATION = 32
NUM_EPOCHS              = 300
LEARNING_RATE           = 3e-5
WEIGHT_DECAY            = 1e-4
WARMUP_RATIO            = 0.1   # fraction of total steps used for warm-up
CLIP_GRAD_NORM          = 1.0
USE_AMP                 = True   # mixed-precision (FP16) — saves ~40% VRAM
CHECKPOINT_EVERY        = 5      # save every N epochs

# Early Stopping
EARLY_STOP_PATIENCE     = 10      # stop if val_loss không cải thiện sau N epochs
EARLY_STOP_MIN_DELTA    = 1e-4   # cải thiện tối thiểu để tính là có tiến bộ

# Backbone fine-tune
FREEZE_VIT_EPOCHS      = 3     # freeze ViT for first N epochs, then unfreeze
FREEZE_PHOBERT_EPOCHS  = 3     # same for PhoBERT

In [11]:
# RL Hyperparameters
RL_LR             = 1e-5       # lower LR than SFT to avoid catastrophic forgetting
RL_EPOCHS         = 5
RL_BATCH_SIZE     = 8
RL_BASELINE_DECAY = 0.99       # EMA decay for reward baseline
RL_ENTROPY_COEF   = 0.01       # entropy bonus to encourage exploration
RL_SAMPLE_SIZE    = 200        # placeholder — overridden after data load

In [12]:
print(f"MODEL_VARIANT  : {MODEL_VARIANT} -> '{VARIANT_DESCRIPTIONS[MODEL_VARIANT]}'", flush=True)
print(f"SKIP_TRAIN     : {SKIP_TRAIN}", flush=True)
print(f"TARGET_CATEGORY: {TARGET_CATEGORY or 'ALL'}", flush=True)
print(f"BATCH_SIZE     : {BATCH_SIZE}  (effective: {BATCH_SIZE * GRADIENT_ACCUMULATION})", flush=True)
print(f"USE_AMP        : {USE_AMP}", flush=True)

MODEL_VARIANT  : A1 -> 'ViT + PhoBERT + Co-Attention + LSTM Decoder'
SKIP_TRAIN     : False
TARGET_CATEGORY: kien_truc
BATCH_SIZE     : 32  (effective: 128)
USE_AMP        : True


## 2. Paths

In [13]:
# Base directories
KAGGLE_DATASET = Path("viet-cultural-vqa")
AUG_PATH = Path("viet-cultural-vqa/augmented_train.json")

WORK           = Path("output")

In [14]:
# Output paths — MODEL_VARIANT in name so parallel runs never collide
CHECKPOINT_DIR  = WORK / "checkpoints"
LOG_DIR         = WORK / "logs"
RESULTS_DIR     = WORK / "results"

CHECKPOINT_BEST   = CHECKPOINT_DIR / f"vqa_best_{MODEL_VARIANT}.pt"
CHECKPOINT_LATEST = CHECKPOINT_DIR / f"vqa_latest_{MODEL_VARIANT}.pt"
LOG_FILE          = LOG_DIR        / f"train_log_{MODEL_VARIANT}.csv"
LOSS_PLOT_FILE    = RESULTS_DIR    / f"loss_curve_{MODEL_VARIANT}.png"
PRED_CSV_FILE     = RESULTS_DIR    / f"predictions_{MODEL_VARIANT}.csv"
METRICS_CSV_FILE  = RESULTS_DIR    / f"metrics_{MODEL_VARIANT}.csv"
ANSWER_VOCAB_FILE = WORK           / "ANSWER_VOCAB.json"   # shared across variants

RL_CHECKPOINT  = CHECKPOINT_DIR / f"vqa_rl_{MODEL_VARIANT}.pt"
RL_METRICS_CSV = RESULTS_DIR    / f"metrics_rl_{MODEL_VARIANT}.csv"

In [15]:
for d in [CHECKPOINT_DIR, LOG_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [16]:
print(f"Working directory  : {WORK.resolve()}", flush=True)
print(f"Checkpoint (best)  : {CHECKPOINT_BEST}", flush=True)
print(f"Log file           : {LOG_FILE}", flush=True)
print(f"Predictions CSV    : {PRED_CSV_FILE}", flush=True)

Working directory  : C:\Users\i_am_fuch\Desktop\deep-learning-final\output
Checkpoint (best)  : output\checkpoints\vqa_best_A1.pt
Log file           : output\logs\train_log_A1.csv
Predictions CSV    : output\results\predictions_A1.csv


## 3. Reproducibility

In [17]:
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [18]:
seed_everything(SEED)

## 4. Dataset

### 4.1 Dataset format

Dataset loaded from local Kaggle Dataset. Each sample contains:

| Field | Description |
|-------|-------------|
| `image_id` | Unique image identifier |
| `image_path` | Relative path to image file |
| `category` | Cultural category |
| `questions` | List of `{question, answer}` dicts |

In [19]:
SPLIT_DIR = f"{KAGGLE_DATASET}/splits"

In [20]:
def load_json_split(filepath, target_category: str = None, max_answer_words: int = None):
    """
    Load a JSON split, optionally filtering by:
      - target_category : keep only records matching this category (case-insensitive)
      - max_answer_words: discard individual QA pairs whose answer has more than N whitespace-tokens
                          (records with no valid QA pairs after filtering are dropped)
    Returns a plain Python list of dicts.
    """
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Unwrap outer dict if needed
    if isinstance(data, dict):
        for key in ["data", "samples", "items", "examples"]:
            if key in data:
                data = data[key]
                break
        else:
            data = next(iter(data.values()))

    flat = []
    for r in data:
        # ── Category filter ──
        cat = str(r.get("category", "")).strip().lower()
        if target_category and cat != target_category.strip().lower():
            continue

        questions_raw = r.get("questions", [])

        # ── Parse questions ──
        if isinstance(questions_raw, str):
            qs = json.loads(questions_raw)
        elif (isinstance(questions_raw, list) and questions_raw
                and isinstance(questions_raw[0], str)):
            qs = json.loads(questions_raw[0])
        else:
            qs = questions_raw

        # ── Answer-length filter: lọc từng QA riêng lẻ, không bỏ cả record ──
        if max_answer_words is not None:
            qs = [qa for qa in qs
                  if len(qa.get("answer", "").strip().split()) <= max_answer_words]

        # Bỏ record nếu không còn QA nào hợp lệ
        if not qs:
            continue

        flat.append({
            "image_id":   str(r.get("image_id", "")),
            "image_path": str(r.get("image_path", "")),
            "category":   str(r.get("category", "")),
            "keyword":    str(r.get("keyword", "")),
            "questions":  qs,  # lưu list đã parse & lọc, không phải raw string
        })

    return flat

In [21]:
TRAIN_DATA = load_json_split(
    f"{SPLIT_DIR}/train_data.json",
    target_category=TARGET_CATEGORY,
    max_answer_words=MAX_ANSWER_WORDS,
)
VAL_DATA = load_json_split(
    f"{SPLIT_DIR}/val_data.json",
    target_category=TARGET_CATEGORY,
    max_answer_words=MAX_ANSWER_WORDS,
)
TEST_DATA = load_json_split(
    f"{SPLIT_DIR}/test_data.json",
    target_category=TARGET_CATEGORY,
    max_answer_words=MAX_ANSWER_WORDS,
)

In [22]:
# ── Load & merge augmented training data ─────────────────────────────────────
def load_augmented_json(filepath, max_answer_words: int = None):
    """
    Load augmented_train.json (flat format: mỗi record là 1 QA pair).
    Convert sang cùng format với TRAIN_DATA:
      {image_id, image_path, category, keyword, questions: [{question, answer}]}
    Các records cùng image_id được gộp lại thành 1 record.
    """
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Filter answer length
    if max_answer_words is not None:
        data = [r for r in data
                if len(r.get("answer", "").strip().split()) <= max_answer_words]

    # Group by image_id → gộp nhiều QA cùng ảnh vào 1 record
    grouped = {}
    for r in data:
        img_id = str(r.get("image_id", ""))
        if img_id not in grouped:
            grouped[img_id] = {
                "image_id":   img_id,
                "image_path": img_id,   # image_id chính là relative path
                "category":   str(r.get("category", "")),
                "keyword":    "",
                "questions":  [],
            }
        grouped[img_id]["questions"].append({
            "question": r.get("question", ""),
            "answer":   r.get("answer", ""),
        })

    result = [r for r in grouped.values() if r["questions"]]
    return result

In [23]:
if AUG_PATH.exists():
    AUG_DATA   = load_augmented_json(AUG_PATH, max_answer_words=MAX_ANSWER_WORDS)
    before_aug = len(TRAIN_DATA)
    TRAIN_DATA = TRAIN_DATA + AUG_DATA
    print(f"Augmented records loaded : {len(AUG_DATA):,}", flush=True)
    print(f"Train before merge       : {before_aug:,} records", flush=True)
    print(f"Train after  merge       : {len(TRAIN_DATA):,} records", flush=True)
else:
    print(f"[WARNING] augmented_train.json not found at {AUG_PATH} — skipping augmentation.", flush=True)

Augmented records loaded : 1,300
Train before merge       : 1,903 records
Train after  merge       : 3,203 records


In [24]:
# Đếm tổng QA pairs thực sự
total_train_qa = sum(len(r["questions"]) for r in TRAIN_DATA)
total_val_qa   = sum(len(r["questions"]) for r in VAL_DATA)
total_test_qa  = sum(len(r["questions"]) for r in TEST_DATA)

print(f"Train : {len(TRAIN_DATA):,} records  →  {total_train_qa:,} QA pairs")
print(f"Val   : {len(VAL_DATA):,} records  →  {total_val_qa:,} QA pairs")
print(f"Test  : {len(TEST_DATA):,} records  →  {total_test_qa:,} QA pairs")
print(f"Total QA pairs: {total_train_qa + total_val_qa + total_test_qa:,}")

Train : 3,203 records  →  5,971 QA pairs
Val   : 407 records  →  871 QA pairs
Test  : 410 records  →  836 QA pairs
Total QA pairs: 7,678


In [25]:
# Override RL_SAMPLE_SIZE based on actual val size
RL_SAMPLE_SIZE = min(100, len(VAL_DATA))
print(f"RL_SAMPLE_SIZE set to: {RL_SAMPLE_SIZE}", flush=True)

RL_SAMPLE_SIZE set to: 100


### 4.2 Answer Vocabulary

In [26]:
PAD_TOKEN = "<pad>"
SOS_TOKEN = "<sos>"
EOS_TOKEN = "<eos>"
UNK_TOKEN = "<unk>"
SPECIAL_TOKENS = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]

In [27]:
def build_answer_vocab(train_records: List[Dict], max_size: int) -> Dict[str, int]:
    """Build answer token vocab from training data."""    
    counter = Counter()
    for r in train_records:
        questions = r.get("questions", "[]")
        if isinstance(questions, str):
            questions = json.loads(questions)
        answer = questions[0]["answer"] if questions else r.get("answer", "")
        tokens = answer.strip().lower().split()
        counter.update(tokens)
    most_common = [w for w, _ in counter.most_common(max_size - len(SPECIAL_TOKENS))]
    vocab = {tok: idx for idx, tok in enumerate(SPECIAL_TOKENS + most_common)}
    return vocab

In [28]:
def save_vocab(vocab: Dict, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(vocab, f, ensure_ascii=False, indent=2)

def load_vocab(path: Path) -> Dict[str, int]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [29]:
if ANSWER_VOCAB_FILE.exists():
    ANSWER_VOCAB = load_vocab(ANSWER_VOCAB_FILE)
    print(f"Loaded answer vocab: {len(ANSWER_VOCAB)} tokens", flush=True)
else:
    ANSWER_VOCAB = build_answer_vocab(TRAIN_DATA, MAX_ANSWER_VOCAB)
    save_vocab(ANSWER_VOCAB, ANSWER_VOCAB_FILE)
    print(f"Built & saved answer vocab: {len(ANSWER_VOCAB)} tokens", flush=True)

IDX2ANSWER = {v: k for k, v in ANSWER_VOCAB.items()}

Loaded answer vocab: 1010 tokens


In [30]:
VOCAB_SIZE = len(ANSWER_VOCAB)
PAD_IDX = ANSWER_VOCAB[PAD_TOKEN]
SOS_IDX = ANSWER_VOCAB[SOS_TOKEN]
EOS_IDX = ANSWER_VOCAB[EOS_TOKEN]
UNK_IDX = ANSWER_VOCAB[UNK_TOKEN]

### 4.3 Image Transforms & VQA Dataset

In [31]:
class VQADataset(Dataset):
    def __init__(
        self,
        records,
        tokenizer,
        answer_vocab: Dict[str, int],
        image_root: Path,
        transform=None,
        max_q_len: int = MAX_QUESTION_LEN,
        max_a_len: int = MAX_ANSWER_LEN,
        augment_answers: bool = False,   # enable only for train split
    ):
        self.records         = records
        self.tokenizer       = tokenizer
        self.answer_vocab    = answer_vocab
        self.image_root      = Path(image_root)
        self.transform       = transform
        self.max_q_len       = max_q_len
        self.max_a_len       = max_a_len
        self.augment_answers = augment_answers

    def __len__(self):
        return len(self.records)

    def _encode_answer(self, answer: str) -> torch.Tensor:
        tokens = answer.strip().lower().split()
        if not tokens:
            tokens = [UNK_TOKEN]
        ids = (
            [SOS_IDX]
            + [self.answer_vocab.get(t, UNK_IDX) for t in tokens[:self.max_a_len]]
            + [EOS_IDX]
        )
        return torch.tensor(ids, dtype=torch.long)

    def _load_image(self, rec) -> Image.Image:
        img_path = rec.get("image_path", "")
        if isinstance(img_path, list):
            img_path = img_path[0] if img_path else ""
        if img_path.startswith("data/"):
            img_path = img_path[len("data/"):]
        full_path = self.image_root.parent / img_path
        if full_path.exists():
            return Image.open(full_path).convert("RGB")
        return Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), color=(128, 128, 128))

    def __getitem__(self, idx):
        rec = self.records[idx]

        image = self._load_image(rec)
        if self.transform:
            image = self.transform(image)

        questions_raw = rec["questions"]
        if isinstance(questions_raw, str):
            questions_list = json.loads(questions_raw)
        elif (isinstance(questions_raw, list) and questions_raw
              and isinstance(questions_raw[0], str)):
            questions_list = json.loads(questions_raw[0])
        else:
            questions_list = questions_raw

        question = questions_list[0]["question"]
        answer   = questions_list[0]["answer"]

        enc = self.tokenizer(
            question,
            max_length=self.max_q_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        input_ids      = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
        answer_ids     = self._encode_answer(answer)

        return {
            "image":          image,
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "answer_ids":     answer_ids,
            "answer_str":     answer,
            "answer_type":    rec.get("answer_type", "unknown"),
            "image_id":       rec["image_id"],
        }


def vqa_collate_fn(batch: List[Dict]) -> Dict:
    images          = torch.stack([b["image"]          for b in batch])
    input_ids       = torch.stack([b["input_ids"]      for b in batch])
    attention_masks = torch.stack([b["attention_mask"] for b in batch])
    answer_ids      = pad_sequence(
        [b["answer_ids"] for b in batch], batch_first=True, padding_value=PAD_IDX
    )
    return {
        "image":          images,
        "input_ids":      input_ids,
        "attention_mask": attention_masks,
        "answer_ids":     answer_ids,
        "answer_str":     [b["answer_str"]  for b in batch],
        "answer_type":    [b["answer_type"] for b in batch],
        "image_id":       [b["image_id"]    for b in batch],
    }

In [32]:
# Load PhoBERT tokenizer
print("Loading PhoBERT tokenizer...", flush=True)
PHOBERT_TOKENIZER = AutoTokenizer.from_pretrained(PHOBERT_MODEL)

Loading PhoBERT tokenizer...


In [33]:
IMAGE_ROOT = KAGGLE_DATASET / "images"

TRAIN_DATASET = VQADataset(TRAIN_DATA, PHOBERT_TOKENIZER,
                            ANSWER_VOCAB, image_root=IMAGE_ROOT,
                            transform=TRAIN_TRANSFORM,
                            augment_answers=True)   # text aug enabled for train
VAL_DATASET   = VQADataset(VAL_DATA,   PHOBERT_TOKENIZER,
                            ANSWER_VOCAB, image_root=IMAGE_ROOT,
                            transform=EVAL_TRANSFORM)
TEST_DATASET  = VQADataset(TEST_DATA,  PHOBERT_TOKENIZER,
                            ANSWER_VOCAB, image_root=IMAGE_ROOT,
                            transform=EVAL_TRANSFORM)

In [34]:
TRAIN_LOADER = DataLoader(TRAIN_DATASET, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=vqa_collate_fn, drop_last=True)
VAL_LOADER   = DataLoader(VAL_DATASET,   batch_size=BATCH_SIZE * 2, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=vqa_collate_fn)
TEST_LOADER  = DataLoader(TEST_DATASET,  batch_size=BATCH_SIZE * 2, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=vqa_collate_fn)

In [35]:
print(f"Train loader : {len(TRAIN_LOADER)} batches", flush=True)
print(f"Val   loader : {len(VAL_LOADER)}   batches", flush=True)
print(f"Test  loader : {len(TEST_LOADER)}  batches", flush=True)

Train loader : 100 batches
Val   loader : 7   batches
Test  loader : 7  batches


## 5. Model Architecture

### 5.1 Image Encoder — ViT

In [36]:
class ViTImageEncoder(nn.Module):
    """
    ViT-Base/16 via timm.
    Outputs patch tokens: (B, num_patches+1, 768) including CLS token.
    """
    def __init__(self, model_name: str = VIT_MODEL, freeze: bool = True):
        super().__init__()
        self.vit     = timm.create_model(model_name, pretrained=True, num_classes=0)
        self.out_dim = self.vit.embed_dim   # 768
        self.set_frozen(freeze)

    def set_frozen(self, freeze: bool):
        for p in self.vit.parameters():
            p.requires_grad = not freeze

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.vit.forward_features(x)   # (B, N+1, 768)

### 5.2 Text Encoder — PhoBERT

In [37]:
class PhoBERTTextEncoder(nn.Module):
    """
    PhoBERT-base-v2 text encoder.
    Returns all token hidden states: (B, seq_len, 768).
    """
    def __init__(self, model_name: str = PHOBERT_MODEL, freeze: bool = True):
        super().__init__()
        self.bert    = AutoModel.from_pretrained(model_name)
        self.out_dim = self.bert.config.hidden_size   # 768
        self.set_frozen(freeze)

    def set_frozen(self, freeze: bool):
        for p in self.bert.parameters():
            p.requires_grad = not freeze

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        return self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state

### 5.3 Co-Attention Fusion

In [38]:
class CoAttentionFusion(nn.Module):
    """
    Guided co-attention:
      - Text attends over image patches  (text-guided visual attention)
      - Image attends over text tokens   (image-guided textual attention)
    Outputs concatenated attended representations projected to (B, FUSION_DIM).
    """
    def __init__(
        self,
        img_dim:    int   = VIT_FEAT_DIM,
        text_dim:   int   = PHOBERT_FEAT_DIM,
        attn_dim:   int   = CO_ATTN_DIM,
        fusion_dim: int   = FUSION_DIM,
        dropout:    float = 0.2,
    ):
        super().__init__()
        self.proj_img  = nn.Linear(img_dim,  attn_dim)
        self.proj_text = nn.Linear(text_dim, attn_dim)

        self.img_attn_query = nn.Linear(attn_dim, attn_dim)
        self.img_attn_key   = nn.Linear(attn_dim, attn_dim)

        self.txt_attn_query = nn.Linear(attn_dim, attn_dim)
        self.txt_attn_key   = nn.Linear(attn_dim, attn_dim)

        self.fusion = nn.Sequential(
            nn.Linear(attn_dim * 2, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.scale = math.sqrt(attn_dim)

    def _attend(self, query_feat, key_feat, value_feat):
        scores   = torch.bmm(query_feat, key_feat.transpose(1, 2)) / self.scale
        weights  = torch.softmax(scores, dim=-1)
        attended = torch.bmm(weights, value_feat).mean(dim=1)
        return attended

    def forward(self, img_feats: torch.Tensor, text_feats: torch.Tensor) -> torch.Tensor:
        V = self.proj_img(img_feats)    # (B, N_img, D)
        T = self.proj_text(text_feats)  # (B, N_txt, D)

        v_att = self._attend(self.img_attn_query(T), self.img_attn_key(V), V)
        t_att = self._attend(self.txt_attn_query(V), self.txt_attn_key(T), T)

        return self.fusion(torch.cat([v_att, t_att], dim=-1))   # (B, FUSION_DIM)

### 5.4 Answer Decoders (A1: LSTM  |  A2: Transformer)

In [39]:
class LSTMDecoder(nn.Module):
    """
    Auto-regressive LSTM decoder.
    Context vector (fusion output) is projected to initial hidden state.
    """
    def __init__(
        self,
        vocab_size:  int   = VOCAB_SIZE,
        embed_dim:   int   = FUSION_DIM,
        hidden_dim:  int   = LSTM_HIDDEN_DIM,
        num_layers:  int   = LSTM_NUM_LAYERS,
        dropout:     float = LSTM_DROPOUT,
        pad_idx:     int   = PAD_IDX,
    ):
        super().__init__()
        self.hidden_dim   = hidden_dim
        self.num_layers   = num_layers
        self.embed        = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm         = nn.LSTM(
            embed_dim, hidden_dim, num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.context_proj = nn.Linear(FUSION_DIM, hidden_dim)
        self.out_proj     = nn.Linear(hidden_dim, vocab_size)
        self.dropout      = nn.Dropout(dropout)

    def _init_hidden(self, context: torch.Tensor):
        h = self.context_proj(context).unsqueeze(0).repeat(self.num_layers, 1, 1)
        c = torch.zeros_like(h)
        return h, c

    def forward(self, context: torch.Tensor, tgt_tokens: torch.Tensor) -> torch.Tensor:
        h, c   = self._init_hidden(context)
        emb    = self.dropout(self.embed(tgt_tokens))
        out, _ = self.lstm(emb, (h, c))
        return self.out_proj(self.dropout(out))

    @torch.no_grad()
    def generate(self, context: torch.Tensor, max_len: int = MAX_ANSWER_LEN) -> torch.Tensor:
        B, device = context.size(0), context.device
        h, c = self._init_hidden(context)
        tok  = torch.full((B, 1), SOS_IDX, dtype=torch.long, device=device)
        outputs = []
        for _ in range(max_len):
            emb        = self.embed(tok)
            out, (h,c) = self.lstm(emb, (h, c))
            tok        = self.out_proj(out.squeeze(1)).argmax(dim=-1, keepdim=True)
            outputs.append(tok)
        return torch.cat(outputs, dim=1)

In [40]:
class TransformerDecoder(nn.Module):
    """
    Standard Transformer decoder with causal mask.
    Memory = context vector expanded as a single-step sequence.
    """
    def __init__(
        self,
        vocab_size:  int   = VOCAB_SIZE,
        d_model:     int   = FUSION_DIM,
        nhead:       int   = TRANSF_NHEAD,
        num_layers:  int   = TRANSF_NUM_LAYERS,
        dim_ff:      int   = TRANSF_FF_DIM,
        dropout:     float = TRANSF_DROPOUT,
        pad_idx:     int   = PAD_IDX,
        max_len:     int   = MAX_ANSWER_LEN + 2,
    ):
        super().__init__()
        self.embed     = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_embed = nn.Embedding(max_len, d_model)
        decoder_layer  = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.decoder   = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.out_proj  = nn.Linear(d_model, vocab_size)
        self.dropout   = nn.Dropout(dropout)

    def _causal_mask(self, T: int, device) -> torch.Tensor:
        return torch.triu(torch.ones(T, T, device=device, dtype=torch.bool), diagonal=1)

    def forward(self, context: torch.Tensor, tgt_tokens: torch.Tensor) -> torch.Tensor:
        B, T = tgt_tokens.shape
        pos  = torch.arange(T, device=tgt_tokens.device).unsqueeze(0)
        emb  = self.dropout(self.embed(tgt_tokens) + self.pos_embed(pos))
        out  = self.decoder(tgt=emb, memory=context.unsqueeze(1),
                            tgt_mask=self._causal_mask(T, tgt_tokens.device))
        return self.out_proj(out)

    @torch.no_grad()
    def generate(self, context: torch.Tensor, max_len: int = MAX_ANSWER_LEN) -> torch.Tensor:
        B, device = context.size(0), context.device
        memory    = context.unsqueeze(1)
        tokens    = torch.full((B, 1), SOS_IDX, dtype=torch.long, device=device)
        for _ in range(max_len):
            pos    = torch.arange(tokens.size(1), device=device).unsqueeze(0)
            emb    = self.embed(tokens) + self.pos_embed(pos)
            out    = self.decoder(tgt=emb, memory=memory,
                                  tgt_mask=self._causal_mask(tokens.size(1), device))
            tokens = torch.cat([tokens, self.out_proj(out[:, -1, :]).argmax(-1, keepdim=True)], dim=1)
        return tokens[:, 1:]   # strip <sos>

### 5.5 Full VQA Model

In [41]:
class VQAModel(nn.Module):
    """
    Full VQA pipeline:
      ViT (timm, inline)  +  PhoBERT (HuggingFace, inline)
      →  Co-Attention Fusion
      →  LSTM decoder (A1) or Transformer decoder (A2)
    """
    def __init__(self, variant: str = MODEL_VARIANT):
        super().__init__()
        self.variant = variant

        self.image_encoder = ViTImageEncoder(freeze=True)
        self.text_encoder  = PhoBERTTextEncoder(freeze=True)
        self.co_attn       = CoAttentionFusion()

        if variant == "A1":
            self.decoder = LSTMDecoder()
        elif variant == "A2":
            self.decoder = TransformerDecoder()
        else:
            raise ValueError(f"Unknown MODEL_VARIANT: {variant!r}. Expected 'A1' or 'A2'.")

        print(f"VQAModel ({variant}) initialised — decoder: {type(self.decoder).__name__}", flush=True)

    def unfreeze_encoders(self):
        self.image_encoder.set_frozen(False)
        self.text_encoder.set_frozen(False)
        print("  [VQAModel] Encoders unfrozen.", flush=True)

    # Aliases for RL compatibility
    def set_frozen_image(self, freeze: bool): self.image_encoder.set_frozen(freeze)
    def set_frozen_text(self,  freeze: bool): self.text_encoder.set_frozen(freeze)

    def encode(self, image, input_ids, attention_mask):
        img_feats  = self.image_encoder(image)
        text_feats = self.text_encoder(input_ids, attention_mask)
        return self.co_attn(img_feats, text_feats)   # (B, FUSION_DIM)

    def forward(self, image, input_ids, attention_mask, answer_ids):
        context    = self.encode(image, input_ids, attention_mask)
        dec_input  = answer_ids[:, :-1]
        dec_target = answer_ids[:, 1:]
        logits     = self.decoder(context, dec_input)
        return logits, dec_target

    @torch.no_grad()
    def generate(self, image, input_ids, attention_mask, max_len=MAX_ANSWER_LEN):
        context = self.encode(image, input_ids, attention_mask)
        return self.decoder.generate(context, max_len=max_len)

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [42]:
MODEL = VQAModel(variant=MODEL_VARIANT).to(DEVICE)
print(f"Trainable params (initial, encoders frozen): {MODEL.count_parameters():,}", flush=True)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


VQAModel (A1) initialised — decoder: LSTMDecoder
Trainable params (initial, encoders frozen): 7,864,306


## 6. Training

### 6.1 Loss & Optimizer

In [43]:
CRITERION = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)

In [44]:
OPTIMIZER = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, MODEL.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

In [45]:
TOTAL_STEPS  = (len(TRAIN_LOADER) // GRADIENT_ACCUMULATION) * NUM_EPOCHS
WARMUP_STEPS = int(TOTAL_STEPS * WARMUP_RATIO)
SCHEDULER    = get_cosine_schedule_with_warmup(
    OPTIMIZER, num_warmup_steps=WARMUP_STEPS, num_training_steps=TOTAL_STEPS
)

print(f"Total optimiser steps : {TOTAL_STEPS}", flush=True)
print(f"Warm-up steps         : {WARMUP_STEPS}", flush=True)

Total optimiser steps : 7500
Warm-up steps         : 750


In [46]:
SCALER = GradScaler(enabled=USE_AMP)

### 6.2 Train & Validation Step

In [47]:
def train_one_epoch(
    model, loader, optimizer, scheduler, scaler, criterion,
    epoch: int, accum_steps: int = GRADIENT_ACCUMULATION,
) -> float:
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(enumerate(loader), total=len(loader),
                desc=f"[{MODEL_VARIANT}] Train Epoch {epoch}", leave=False)

    for step, batch in pbar:
        image          = batch["image"].to(DEVICE, non_blocking=True)
        input_ids      = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        answer_ids     = batch["answer_ids"].to(DEVICE, non_blocking=True)

        with autocast(enabled=USE_AMP):
            logits, targets = model(image, input_ids, attention_mask, answer_ids)
            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
            ) / accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % accum_steps == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            torch.cuda.empty_cache()

        total_loss += loss.item() * accum_steps
        pbar.set_postfix(loss=f"{loss.item() * accum_steps:.4f}")

    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, criterion) -> float:
    model.eval()
    total_loss = 0.0

    pbar = tqdm(loader, total=len(loader), desc=f"[{MODEL_VARIANT}] Val", leave=False)
    for batch in pbar:
        image          = batch["image"].to(DEVICE, non_blocking=True)
        input_ids      = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        answer_ids     = batch["answer_ids"].to(DEVICE, non_blocking=True)

        with autocast(enabled=USE_AMP):
            logits, targets = model(image, input_ids, attention_mask, answer_ids)
            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
            )

        total_loss += loss.item()
        torch.cuda.empty_cache()

    return total_loss / len(loader)

### 6.3 Checkpoint Helpers

In [48]:
def save_checkpoint(model, optimizer, scheduler, scaler, epoch, val_loss, path: Path):
    torch.save({
        "epoch":        epoch,
        "variant":      MODEL_VARIANT,
        "model_state":  model.state_dict(),
        "optim_state":  optimizer.state_dict(),
        "sched_state":  scheduler.state_dict(),
        "scaler_state": scaler.state_dict(),
        "val_loss":     val_loss,
    }, path)
    print(f"  [Checkpoint] Saved -> {path}", flush=True)


def load_checkpoint(model, optimizer, scheduler, scaler, path: Path):
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optim_state"])
    scheduler.load_state_dict(ckpt["sched_state"])
    scaler.load_state_dict(ckpt["scaler_state"])
    print(f"  [Checkpoint] Loaded from {path}  (epoch={ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f})",
          flush=True)
    return ckpt["epoch"], ckpt["val_loss"]

### 6.4 Main Training Loop

In [49]:
HISTORY = {"train_loss": [], "val_loss": []}

In [ ]:
if SKIP_TRAIN:
    if not CHECKPOINT_BEST.exists():
        raise FileNotFoundError(f"SKIP_TRAIN=True but no checkpoint found at {CHECKPOINT_BEST}")
    load_checkpoint(MODEL, OPTIMIZER, SCHEDULER, SCALER, CHECKPOINT_BEST)
    print("Skipping training — checkpoint loaded.", flush=True)

else:
    start_epoch   = 1
    best_val_loss = float("inf")

    if CHECKPOINT_LATEST.exists():
        start_epoch, best_val_loss = load_checkpoint(
            MODEL, OPTIMIZER, SCHEDULER, SCALER, CHECKPOINT_LATEST
        )
        start_epoch += 1
        print(f"  Resuming from epoch {start_epoch}", flush=True)

    es_counter   = 0
    es_best_loss = best_val_loss
    log_rows     = []

    for epoch in range(start_epoch, NUM_EPOCHS + 1):
        # Scheduled backbone unfreezing
        if epoch == FREEZE_VIT_EPOCHS + 1:
            MODEL.unfreeze_encoders()
            OPTIMIZER = torch.optim.AdamW(
                MODEL.parameters(), lr=LEARNING_RATE * 0.1, weight_decay=WEIGHT_DECAY
            )
            print(f"  [Epoch {epoch}] Encoders unfrozen with lr={LEARNING_RATE * 0.1:.2e}", flush=True)

        t0         = time.time()
        train_loss = train_one_epoch(MODEL, TRAIN_LOADER, OPTIMIZER, SCHEDULER, SCALER, CRITERION, epoch)
        val_loss   = validate(MODEL, VAL_LOADER, CRITERION)
        elapsed    = time.time() - t0

        HISTORY["train_loss"].append(train_loss)
        HISTORY["val_loss"].append(val_loss)
        log_rows.append({"epoch": epoch, "train_loss": train_loss,
                         "val_loss": val_loss, "time_s": elapsed})

        print(f"Epoch {epoch:03d}/{NUM_EPOCHS}  "
              f"train={train_loss:.4f}  val={val_loss:.4f}  "
              f"({elapsed:.0f}s)  [{MODEL_VARIANT}]", flush=True)

        # Save latest (always)
        save_checkpoint(MODEL, OPTIMIZER, SCHEDULER, SCALER, epoch, val_loss, CHECKPOINT_LATEST)

        # Save best (only when improved)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(MODEL, OPTIMIZER, SCHEDULER, SCALER, epoch, val_loss, CHECKPOINT_BEST)
            print(f"  ★ New best val_loss={best_val_loss:.4f}", flush=True)

        # Early stopping
        if es_best_loss - val_loss > EARLY_STOP_MIN_DELTA:
            es_best_loss = val_loss
            es_counter   = 0
        else:
            es_counter += 1
            print(f"  [EarlyStopping] No improvement for {es_counter}/{EARLY_STOP_PATIENCE} epochs", flush=True)
            if es_counter >= EARLY_STOP_PATIENCE:
                print(f"  [EarlyStopping] Triggered at epoch {epoch} — stopping training.", flush=True)
                break

        gc.collect()
        torch.cuda.empty_cache()

    pd.DataFrame(log_rows).to_csv(LOG_FILE, index=False)
    print(f"Training log saved -> {LOG_FILE}", flush=True)

## 7. Loss Curves

In [ ]:
def plot_loss_curves(history: Dict, save_path: Path):
    if not history["train_loss"]:
        print("No training history to plot — run training first.", flush=True)
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(f"Training Curves — Variant {MODEL_VARIANT}", fontsize=13, fontweight="bold")

    for ax, key, title, color in zip(
        axes,
        ["train_loss", "val_loss"],
        ["Train Loss",  "Val Loss"],
        ["steelblue",   "tomato"],
    ):
        ax.plot(range(1, len(history[key]) + 1), history[key], lw=2, color=color)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.grid(alpha=0.3)

    plt.tight_layout()
    fig.savefig(save_path, dpi=120, bbox_inches="tight")
    print(f"Loss curves saved -> {save_path}", flush=True)
    plt.show()

In [ ]:
plot_loss_curves(HISTORY, LOSS_PLOT_FILE)

## 8. Evaluation

### 8.1 Metric Functions

| Metric | Description |
|--------|-------------|
| **VQA Exact Match** | Prediction == reference (case-insensitive) |
| **VQA Soft Accuracy** | Token-F1 proxy for partial credit |
| **BLEU-1/4** | N-gram overlap |
| **ROUGE-L** | Longest common subsequence |
| **METEOR** | Alignment-based metric with stemming/synonym matching |
| **BERTScore** | Semantic similarity via embeddings |
| **LLM-as-a-judge** | Claude scores prediction correctness 0–1 given question + reference |

In [ ]:
def decode_ids(ids: torch.Tensor, vocab: Dict[int, str]) -> str:
    """Convert token id tensor to string, stopping at EOS."""    
    tokens = []
    for t in ids.tolist():
        if t == EOS_IDX:
            break
        if t not in (PAD_IDX, SOS_IDX):
            tokens.append(vocab.get(t, UNK_TOKEN))
    return " ".join(tokens) if tokens else ""

In [ ]:
def vqa_exact_match(preds: List[str], refs: List[str]) -> float:
    correct = sum(p.strip().lower() == r.strip().lower() for p, r in zip(preds, refs))
    return correct / len(refs) if refs else 0.0


def vqa_soft_accuracy(preds: List[str], refs: List[str]) -> float:
    """Token-F1 proxy for VQA soft accuracy."""    
    scores = []
    for pred, ref in zip(preds, refs):
        pred_toks = set(pred.strip().lower().split())
        ref_toks  = set(ref.strip().lower().split())
        if not pred_toks and not ref_toks:
            scores.append(1.0)
        elif not pred_toks or not ref_toks:
            scores.append(0.0)
        else:
            overlap   = len(pred_toks & ref_toks)
            precision = overlap / len(pred_toks)
            recall    = overlap / len(ref_toks)
            f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
            scores.append(min(f1, 1.0))
    return float(np.mean(scores))


def compute_bleu(preds: List[str], refs: List[str]) -> Dict[str, float]:
    bleu1 = BLEU(max_ngram_order=1)
    bleu4 = BLEU(max_ngram_order=4)
    refs_wrapped = [[r] for r in refs]
    return {
        "bleu1": bleu1.corpus_score(preds, list(zip(*refs_wrapped))).score / 100,
        "bleu4": bleu4.corpus_score(preds, list(zip(*refs_wrapped))).score / 100,
    }


def compute_rouge(preds: List[str], refs: List[str]) -> float:
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    return float(np.mean([scorer.score(r, p)["rougeL"].fmeasure for p, r in zip(preds, refs)]))


def compute_meteor(preds: List[str], refs: List[str]) -> float:
    return float(np.mean([nltk_meteor([r.split()], p.split()) for p, r in zip(preds, refs)]))


def compute_bertscore(preds: List[str], refs: List[str]) -> float:
    if not HAS_BERTSCORE or not preds:
        return 0.0
    _, _, F = bert_score.score(preds, refs, lang="vi", verbose=False)
    return float(F.mean().item())


def compute_llm_judge(
    preds: List[str],
    refs:  List[str],
    questions: Optional[List[str]] = None,
    sample_size: int = 100,
    model: str = "claude-sonnet-4-20250514",
) -> float:
    """Uses Claude as a judge to score prediction quality (0–1)."""
    try:
        client = anthropic.Anthropic()
    except Exception as e:
        print(f"  [LLM-judge] Anthropic client error: {e}", flush=True)
        return 0.0

    indices = list(range(len(preds)))
    if len(indices) > sample_size:
        random.shuffle(indices)
        indices = indices[:sample_size]

    scores = []
    for idx in tqdm(indices, desc="[LLM-judge]", leave=False):
        q    = questions[idx] if questions else "N/A"
        prompt = (
            f"You are evaluating a Vietnamese Visual Question Answering system.\n"
            f"Question  : {q}\n"
            f"Reference : {refs[idx]}\n"
            f"Prediction: {preds[idx]}\n\n"
            "Score the prediction from 0.0 to 1.0 based on semantic correctness "
            "compared to the reference. "
            "1.0 = perfectly correct, 0.5 = partially correct, 0.0 = wrong.\n"
            "Reply with ONLY a number between 0.0 and 1.0, nothing else."
        )
        try:
            resp  = client.messages.create(
                model=model, max_tokens=16,
                messages=[{"role": "user", "content": prompt}],
            )
            score = float(resp.content[0].text.strip())
            score = max(0.0, min(1.0, score))
        except Exception:
            score = 0.0
        scores.append(score)

    return float(np.mean(scores)) if scores else 0.0


def compute_all_metrics(
    preds: List[str],
    refs:  List[str],
    questions: Optional[List[str]] = None,
    run_llm_judge: bool = False,
) -> Dict[str, float]:
    metrics = {}
    metrics["vqa_exact"]  = vqa_exact_match(preds, refs)
    metrics["vqa_soft"]   = vqa_soft_accuracy(preds, refs)
    metrics.update(compute_bleu(preds, refs))
    metrics["rouge_l"]    = compute_rouge(preds, refs)
    metrics["meteor"]     = compute_meteor(preds, refs)
    metrics["bert_score"] = compute_bertscore(preds, refs)
    if run_llm_judge:
        metrics["llm_judge"] = compute_llm_judge(preds, refs, questions)
    return metrics

### 8.2 Inference & Full Evaluation

In [ ]:
@torch.no_grad()
def run_inference(model, loader, split_name: str = "test") -> pd.DataFrame:
    model.eval()
    rows = []
    pbar = tqdm(loader, total=len(loader),
                desc=f"[{MODEL_VARIANT}] Inference ({split_name})", leave=True)
    for batch in pbar:
        image          = batch["image"].to(DEVICE, non_blocking=True)
        input_ids      = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)

        with autocast(enabled=USE_AMP):
            pred_ids = model.generate(image, input_ids, attention_mask)

        for i in range(len(batch["image_id"])):
            pred_str = decode_ids(pred_ids[i], IDX2ANSWER)
            q_ids    = batch["input_ids"][i]
            q_str    = PHOBERT_TOKENIZER.decode(
                q_ids[q_ids != PHOBERT_TOKENIZER.pad_token_id],
                skip_special_tokens=True,
            )
            rows.append({
                "image_id":    batch["image_id"][i],
                "question":    q_str,
                "prediction":  pred_str,
                "reference":   batch["answer_str"][i],
                "answer_type": batch["answer_type"][i],
            })
        torch.cuda.empty_cache()

    return pd.DataFrame(rows)


def evaluate_and_report(
    model,
    loader,
    split_name:    str  = "test",
    pred_csv:      Path = PRED_CSV_FILE,
    metrics_csv:   Path = METRICS_CSV_FILE,
    run_llm_judge: bool = LLM_JUDGE_EVAL,
) -> Dict[str, float]:
    if CHECKPOINT_BEST.exists():
        ckpt = torch.load(CHECKPOINT_BEST, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state"])
        print(f"  [Eval] Loaded best checkpoint (epoch={ckpt['epoch']})", flush=True)

    df = run_inference(model, loader, split_name)
    df.to_csv(pred_csv, index=False, encoding="utf-8")
    print(f"Predictions saved -> {pred_csv}", flush=True)

    preds     = df["prediction"].tolist()
    refs      = df["reference"].tolist()
    questions = df["question"].tolist()

    metrics  = compute_all_metrics(preds, refs, questions, run_llm_judge=run_llm_judge)
    per_type = {
        atype: vqa_exact_match(grp["prediction"].tolist(), grp["reference"].tolist())
        for atype, grp in df.groupby("answer_type")
    }

    print(f"\n{'='*52}", flush=True)
    print(f"  Evaluation Results — Variant {MODEL_VARIANT}  ({split_name})", flush=True)
    print(f"{'='*52}", flush=True)
    for k, v in metrics.items():
        print(f"  {k:15s}: {v*100:.2f}%", flush=True)
    print(f"  Per answer type (exact match):", flush=True)
    for atype, acc in per_type.items():
        print(f"    {atype:12s}: {acc*100:.2f}%", flush=True)
    print(f"{'='*52}\n", flush=True)

    metrics_row = {"variant": MODEL_VARIANT, "split": split_name,
                   **{k: round(v*100, 2) for k, v in metrics.items()}}
    metrics_row.update({f"acc_type_{k}": round(v*100, 2) for k, v in per_type.items()})
    pd.DataFrame([metrics_row]).to_csv(metrics_csv, index=False)
    print(f"Metrics saved -> {metrics_csv}", flush=True)

    return metrics

In [ ]:
TEST_METRICS = evaluate_and_report(MODEL, TEST_LOADER, split_name="test")

## 9. Results Visualisation

In [ ]:
def plot_metrics_bar(metrics: Dict[str, float], variant: str):
    names  = list(metrics.keys())
    values = [v * 100 for v in metrics.values()]

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar(names, values, color="steelblue", edgecolor="white", linewidth=0.8)
    ax.set_ylim(0, 105)
    ax.set_ylabel("Score (%)")
    ax.set_title(f"Evaluation Metrics — {variant}", fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f"{val:.1f}", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    fig.savefig(RESULTS_DIR / f"metrics_bar_{MODEL_VARIANT}.png", dpi=120, bbox_inches="tight")
    plt.show()

In [ ]:
plot_metrics_bar(TEST_METRICS, MODEL_VARIANT)

In [ ]:
def show_qualitative(pred_csv: Path, dataset_records, n: int = 8):
    """Display N random samples with image, prediction, and reference answer."""    
    df      = pd.read_csv(pred_csv)
    samples = df.sample(min(n, len(df)), random_state=SEED)

    id_to_img_path = {}
    for rec in dataset_records:
        img_path = rec.get("image_path", "")
        if img_path.startswith("data/"):
            img_path = img_path[len("data/"):]
        full = IMAGE_ROOT.parent / img_path
        if full.exists():
            id_to_img_path[rec["image_id"]] = full

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle(f"Qualitative Examples — {MODEL_VARIANT}", fontsize=13, fontweight="bold")
    axes = axes.flatten()

    for ax, (_, row) in zip(axes, samples.iterrows()):
        img_id = row["image_id"]
        if img_id in id_to_img_path:
            ax.imshow(Image.open(id_to_img_path[img_id]).convert("RGB"))
        else:
            ax.set_facecolor("#f0f0f0")
            ax.text(0.5, 0.5, "img not found", ha="center", va="center",
                    transform=ax.transAxes, fontsize=8)

        color = "green" if row["prediction"].strip().lower() == row["reference"].strip().lower() else "red"
        ax.set_title(
            f"Pred: {row['prediction']}\nRef : {row['reference']}",
            fontsize=8, color=color, pad=4,
        )
        ax.axis("off")

    plt.tight_layout()
    fig.savefig(RESULTS_DIR / f"qualitative_{MODEL_VARIANT}.png", dpi=120, bbox_inches="tight")
    plt.show()

In [ ]:
show_qualitative(PRED_CSV_FILE, TEST_DATA)

## 10. Cross-Variant Comparison (A1 vs A2)

In [ ]:
def load_metrics_csv(variant: str) -> Optional[pd.DataFrame]:
    path = RESULTS_DIR / f"metrics_{variant}.csv"
    if path.exists():
        return pd.read_csv(path)
    print(f"  [Compare] No metrics file for {variant} at {path}", flush=True)
    return None


def compare_variants():
    """
    Reads A1 and A2 metrics CSVs and draws a grouped bar chart.
    Run only after both variants have been evaluated.
    """
    dfs = [df for df in (load_metrics_csv(v) for v in ["A1", "A2"]) if df is not None]

    if len(dfs) < 2:
        print("Run both A1 and A2 notebooks first, then re-run this cell.", flush=True)
        return

    combined    = pd.concat(dfs, ignore_index=True)
    metric_cols = [c for c in combined.columns if c not in ["variant", "split"]]

    x, width = np.arange(len(metric_cols)), 0.35
    fig, ax  = plt.subplots(figsize=(14, 5))
    colors   = ["steelblue", "tomato"]

    for i, (_, row) in enumerate(combined.iterrows()):
        vals = [row[c] for c in metric_cols]
        bars = ax.bar(x + i * width - width / 2, vals, width,
                      label=row["variant"], color=colors[i % 2], alpha=0.85)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                    f"{v:.1f}", ha="center", va="bottom", fontsize=7)

    ax.set_xticks(x)
    ax.set_xticklabels(metric_cols, rotation=15, ha="right")
    ax.set_ylabel("Score (%)")
    ax.set_title("A1 (LSTM decoder) vs A2 (Transformer decoder)", fontweight="bold")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    fig.savefig(RESULTS_DIR / "comparison_A1_vs_A2.png", dpi=120, bbox_inches="tight")
    plt.show()

    print(combined[["variant"] + metric_cols].to_string(index=False), flush=True)

In [ ]:
compare_variants()

## 11. Reinforcement Learning Fine-tuning (RLHF / REINFORCE)

### 11.1 Overview

We fine-tune the trained VQA model using **REINFORCE** (policy gradient) with VQA soft accuracy as the reward signal.

| Component | Detail |
|-----------|--------|
| **Policy** | Trained VQA model (frozen encoders, trainable decoder) |
| **Reward** | Token-F1 between generated answer and reference |
| **Baseline** | Exponential moving average of past rewards |
| **Preference data** | Up to `RL_SAMPLE_SIZE` pairs sampled from val set |
| **Comparison** | RL-tuned vs SFT (base trained model) on all metrics |

In [ ]:
def compute_reward(pred: str, ref: str) -> float:
    """Token-F1 reward in [0, 1]."""    
    pred_toks = set(pred.strip().lower().split())
    ref_toks  = set(ref.strip().lower().split())
    if not pred_toks and not ref_toks:
        return 1.0
    if not pred_toks or not ref_toks:
        return 0.0
    overlap   = len(pred_toks & ref_toks)
    precision = overlap / len(pred_toks)
    recall    = overlap / len(ref_toks)
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return min(f1, 1.0)

In [ ]:
def reinforce_step(
    model,
    batch: Dict,
    optimizer,
    baseline: float,
    entropy_coef: float = RL_ENTROPY_COEF,
    max_len: int = MAX_ANSWER_LEN,
) -> Tuple[float, float, float]:
    """One REINFORCE update step. Returns (loss, mean_reward, new_baseline)."""    
    model.train()
    image          = batch["image"].to(DEVICE)
    input_ids      = batch["input_ids"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)
    refs           = batch["answer_str"]
    B              = image.size(0)

    with autocast(enabled=USE_AMP):
        context = model.encode(image, input_ids, attention_mask)

    log_probs_list, entropy_list = [], []
    tokens = torch.full((B, 1), SOS_IDX, dtype=torch.long, device=DEVICE)

    if MODEL_VARIANT == "A1":
        h, c = model.decoder._init_hidden(context)
    else:
        memory = context.unsqueeze(1)

    for step in range(max_len):
        with autocast(enabled=USE_AMP):
            if MODEL_VARIANT == "A1":
                emb        = model.decoder.embed(tokens[:, -1:])
                out, (h,c) = model.decoder.lstm(emb, (h, c))
                logit      = model.decoder.out_proj(out.squeeze(1))
            else:
                pos   = torch.arange(tokens.size(1), device=DEVICE).unsqueeze(0)
                emb   = model.decoder.embed(tokens) + model.decoder.pos_embed(pos)
                mask  = model.decoder._causal_mask(tokens.size(1), DEVICE)
                out   = model.decoder.decoder(tgt=emb, memory=memory, tgt_mask=mask)
                logit = model.decoder.out_proj(out[:, -1, :])

        dist    = torch.distributions.Categorical(torch.softmax(logit, dim=-1))
        sampled = dist.sample()
        log_probs_list.append(dist.log_prob(sampled))
        entropy_list.append(dist.entropy())
        tokens = torch.cat([tokens, sampled.unsqueeze(1)], dim=1)

    preds   = [decode_ids(tokens[i, 1:], IDX2ANSWER) for i in range(B)]
    rewards = torch.tensor(
        [compute_reward(p, r) for p, r in zip(preds, refs)],
        dtype=torch.float32, device=DEVICE,
    )

    log_probs = torch.stack(log_probs_list, dim=1)
    entropy   = torch.stack(entropy_list,   dim=1)
    advantage = (rewards - baseline).unsqueeze(1)

    loss = -(advantage.detach() * log_probs).mean() - entropy_coef * entropy.mean()

    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD_NORM)
    optimizer.step()

    mean_reward  = rewards.mean().item()
    new_baseline = RL_BASELINE_DECAY * baseline + (1 - RL_BASELINE_DECAY) * mean_reward
    return loss.item(), mean_reward, new_baseline

In [ ]:
def train_rl(
    model,
    loader,
    num_epochs: int = RL_EPOCHS,
    lr: float       = RL_LR,
) -> Dict:
    """Fine-tune with REINFORCE. Only decoder parameters are updated."""    
    model.set_frozen_image(True)
    model.set_frozen_text(True)
    for p in model.decoder.parameters():
        p.requires_grad = True

    rl_optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr
    )

    baseline   = 0.0
    rl_history = {"loss": [], "reward": []}

    for epoch in range(1, num_epochs + 1):
        epoch_losses, epoch_rewards = [], []
        pbar = tqdm(loader, desc=f"[RL {MODEL_VARIANT}] Epoch {epoch}", leave=False)
        for batch in pbar:
            loss, reward, baseline = reinforce_step(model, batch, rl_optimizer, baseline)
            epoch_losses.append(loss)
            epoch_rewards.append(reward)
            pbar.set_postfix(reward=f"{reward:.3f}", loss=f"{loss:.4f}")

        mean_loss   = np.mean(epoch_losses)
        mean_reward = np.mean(epoch_rewards)
        rl_history["loss"].append(mean_loss)
        rl_history["reward"].append(mean_reward)
        print(f"RL Epoch {epoch:02d}/{num_epochs}  "
              f"loss={mean_loss:.4f}  reward={mean_reward:.4f}  "
              f"baseline={baseline:.4f}  [{MODEL_VARIANT}]", flush=True)
        gc.collect()
        torch.cuda.empty_cache()

    torch.save({"model_state": model.state_dict(), "variant": MODEL_VARIANT}, RL_CHECKPOINT)
    print(f"RL checkpoint saved -> {RL_CHECKPOINT}", flush=True)
    return rl_history

In [ ]:
# Load best SFT checkpoint before RL fine-tuning
if CHECKPOINT_BEST.exists():
    ckpt = torch.load(CHECKPOINT_BEST, map_location=DEVICE)
    MODEL.load_state_dict(ckpt["model_state"])
    print(f"Loaded SFT checkpoint for RL init (epoch={ckpt['epoch']})", flush=True)

In [ ]:
RL_HISTORY = train_rl(MODEL, TRAIN_LOADER)

In [ ]:
def plot_rl_curves(rl_history: Dict):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"RL Training Curves — {MODEL_VARIANT}", fontweight="bold")

    axes[0].plot(rl_history["reward"], color="green", lw=2)
    axes[0].set_title("Mean Reward per Epoch")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Reward (token-F1)")
    axes[0].grid(alpha=0.3)

    axes[1].plot(rl_history["loss"], color="orange", lw=2)
    axes[1].set_title("REINFORCE Loss per Epoch")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    fig.savefig(RESULTS_DIR / f"rl_curves_{MODEL_VARIANT}.png", dpi=120, bbox_inches="tight")
    plt.show()

In [ ]:
plot_rl_curves(RL_HISTORY)

### 11.2 RL vs SFT Comparison

In [ ]:
def compare_rl_vs_sft(model):
    """Evaluates model under SFT and RL weights and prints a side-by-side table."""    
    results = {}

    for label, ckpt_path in [("SFT", CHECKPOINT_BEST), ("RL", RL_CHECKPOINT)]:
        if not ckpt_path.exists():
            print(f"  [{label}] checkpoint not found at {ckpt_path}", flush=True)
            continue
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state"])
        print(f"\nEvaluating {label}...", flush=True)

        pred_csv    = RESULTS_DIR / f"predictions_{MODEL_VARIANT}_{label}.csv"
        metrics_csv = RESULTS_DIR / f"metrics_{MODEL_VARIANT}_{label}.csv"
        metrics     = evaluate_and_report(
            model, TEST_LOADER,
            split_name=f"test_{label}",
            pred_csv=pred_csv,
            metrics_csv=metrics_csv,
            run_llm_judge=LLM_JUDGE_EVAL,
        )
        results[label] = metrics

    if len(results) == 2:
        print(f"\n{'='*60}", flush=True)
        print(f"  SFT vs RL — {MODEL_VARIANT}", flush=True)
        print(f"{'='*60}", flush=True)
        header = f"{'Metric':<18} {'SFT':>8} {'RL':>8} {'Delta':>8}"
        print(header, flush=True)
        print("-" * len(header), flush=True)
        for k in results["SFT"]:
            sft_val = results["SFT"].get(k, 0) * 100
            rl_val  = results["RL"].get(k, 0)  * 100
            delta   = rl_val - sft_val
            print(f"  {k:<16} {sft_val:>8.2f} {rl_val:>8.2f} {'+' if delta >= 0 else ''}{delta:>7.2f}",
                  flush=True)
        print(f"{'='*60}\n", flush=True)

        metric_names = list(results["SFT"].keys())
        sft_vals = [results["SFT"][k] * 100 for k in metric_names]
        rl_vals  = [results["RL"][k]  * 100 for k in metric_names]
        x, w     = np.arange(len(metric_names)), 0.35
        fig, ax  = plt.subplots(figsize=(13, 5))
        ax.bar(x - w/2, sft_vals, w, label="SFT", color="steelblue", alpha=0.85)
        ax.bar(x + w/2, rl_vals,  w, label="RL",  color="seagreen",  alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels(metric_names, rotation=15, ha="right")
        ax.set_ylabel("Score (%)")
        ax.set_title(f"SFT vs RL Fine-tuning — {MODEL_VARIANT}", fontweight="bold")
        ax.legend()
        ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        fig.savefig(RESULTS_DIR / f"sft_vs_rl_{MODEL_VARIANT}.png", dpi=120, bbox_inches="tight")
        plt.show()

    return results

In [ ]:
rl_comparison = compare_rl_vs_sft(MODEL)